# Garhwali ASR baseline: existing VAANI reference labels

This notebook is an end-to-end reproducibility example using the private Kaggle v2 copy of 2,718 existing CC BY 4.0 ARTPARK-IISc/VAANI audio/reference pairs: 2,202 train / 373 validation / 143 test. It adds no recordings or transcripts. The labels have not been adjudicated by a native speaker. It fine-tunes `openai/whisper-tiny` for one epoch and reports validation WER/CER. The test split is not scored by default; it has already been consulted during project development and is not a blind benchmark. The model is saved to Kaggle working storage and is not published automatically.

Enable Kaggle Internet to fetch the Whisper Tiny base weights (and to use the `kagglehub` fallback). If `transformers` is missing in the Kaggle image, install the version in `requirements-asr.txt` before running the training cell. CPU works but is slower; a free Kaggle GPU can shorten the run when available.

Source: https://huggingface.co/datasets/ARTPARK-IISc/Vaani


In [ ]:
import csv, hashlib, json, os, random, re, unicodedata, wave, zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import torch

SEED = 17
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
ROOT = Path('/kaggle/input')
EXPECTED_SPLITS = {'train': 2202, 'validation': 373, 'test': 143}

def inspect_manifest(path):
    with path.open(encoding='utf-8', newline='') as stream:
        candidate_rows = list(csv.DictReader(stream))
    if not candidate_rows or not {'split', 'audio_file', 'audio_sha256', 'transcript'} <= set(candidate_rows[0]):
        return None
    candidate_splits = {name: [row for row in candidate_rows if row['split'] == name] for name in EXPECTED_SPLITS}
    if {name: len(values) for name, values in candidate_splits.items()} != EXPECTED_SPLITS:
        return None
    if len({row['audio_sha256'] for row in candidate_rows}) != len(candidate_rows):
        return None
    if not all((path.parent / row['audio_file']).is_file() for row in candidate_rows):
        return None
    return candidate_rows

matches = [(path.parent, path, inspect_manifest(path)) for path in sorted(ROOT.rglob('manifest.csv'))]
matches = [item for item in matches if item[2] is not None]
if len(matches) > 1:
    hashes = {hashlib.sha256(item[1].read_bytes()).hexdigest() for item in matches}
    if len(hashes) > 1:
        raise RuntimeError('More than one different 2,718-row ASR manifest is attached; remove the duplicate or choose one dataset input.')
if matches:
    DATA, manifest_path, rows = matches[0]
else:
    zips = sorted(ROOT.rglob('garhwali-asr-reference-v0.2.zip'))
    if len(zips) == 1:
        DATA = Path('/kaggle/working/reference_input')
        DATA.mkdir(exist_ok=True)
        with zipfile.ZipFile(zips[0]) as archive: archive.extractall(DATA)
        manifest_path = DATA / 'manifest.csv'
        rows = inspect_manifest(manifest_path)
    elif len(zips) > 1:
        raise RuntimeError('More than one ASR dataset archive is attached; keep one input.')
    else:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download('rushilrawat1/garhwali-asr-reference-clips-v0-1'))
        manifest_path = downloaded / 'manifest.csv'
        DATA = downloaded
        rows = inspect_manifest(manifest_path)
if rows is None:
    raise RuntimeError('Could not find the expected v2 ASR manifest and all audio files. Attach the Kaggle ASR dataset input and check its version.')
splits = {name: [row for row in rows if row['split'] == name] for name in EXPECTED_SPLITS}
assert len({row['audio_sha256'] for row in rows}) == len(rows)
MANIFEST_SHA256 = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
print('Input:', DATA, 'rows:', {name: len(values) for name, values in splits.items()}, 'manifest SHA-256:', MANIFEST_SHA256, 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = 'openai/whisper-tiny'
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
MODEL_REVISION = getattr(model.config, '_commit_hash', None)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.config.use_cache = False
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

def read_wav(row):
    with wave.open(str(DATA / row['audio_file']), 'rb') as stream:
        assert stream.getframerate() == 16000 and stream.getnchannels() == 1 and stream.getsampwidth() == 2
        audio = np.frombuffer(stream.readframes(stream.getnframes()), dtype='<i2').astype(np.float32)/32768.0
    return audio

def features_and_labels(row):
    features = processor(read_wav(row), sampling_rate=16000, return_tensors='pt').input_features.to(device)
    labels = processor.tokenizer(row['transcript'], return_tensors='pt').input_ids.to(device)
    return features, labels

order = list(splits['train']); random.shuffle(order)
model.train(); losses=[]
for step, row in enumerate(order, 1):
    features, labels = features_and_labels(row)
    optimizer.zero_grad(set_to_none=True)
    loss = model(input_features=features, labels=labels).loss
    loss.backward(); optimizer.step()
    losses.append(float(loss.detach().cpu()))
    if step == 1 or step % 100 == 0:
        print(f'step {step}/{len(order)} loss {losses[-1]:.4f}', flush=True)
OUT = Path('/kaggle/working/garhwali-whisper-tiny-reference-v0.2')
OUT.mkdir(exist_ok=True)
model.save_pretrained(OUT); processor.save_pretrained(OUT)
print('Training complete; mean loss',sum(losses)/len(losses))


In [ ]:
def normalize(text):
    text = unicodedata.normalize('NFKC', text).casefold()
    text = ''.join(ch if (ch.isspace() or unicodedata.category(ch)[0] in ('L','M','N')) else ' ' for ch in text)
    return ' '.join(text.split())

def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        current = [i] + [0] * len(b)
        for j, y in enumerate(b, 1):
            current[j] = min(current[j - 1] + 1, previous[j] + 1, previous[j - 1] + (x != y))
        previous = current
    return previous[-1]

def score(split):
    model.eval(); totals = Counter(); predictions = []
    with torch.no_grad():
        for row in splits[split]:
            features = processor(read_wav(row), sampling_rate=16000, return_tensors='pt').input_features.to(device)
            tokens = model.generate(features, language='hi', task='transcribe', max_length=128)
            hypothesis = processor.batch_decode(tokens, skip_special_tokens=True)[0].strip()
            reference = row['transcript']
            ref, hyp = normalize(reference), normalize(hypothesis)
            totals['word_errors'] += edit_distance(ref.split(), hyp.split())
            totals['reference_words'] += len(ref.split())
            totals['character_errors'] += edit_distance(list(ref.replace(' ', '')), list(hyp.replace(' ', '')))
            totals['reference_characters'] += len(ref.replace(' ', ''))
            totals['rows'] += 1
            predictions.append({'audio_sha256': row['audio_sha256'], 'split': split, 'reference': reference, 'hypothesis': hypothesis})
    result = dict(totals)
    result['wer'] = totals['word_errors'] / max(1, totals['reference_words'])
    result['cer'] = totals['character_errors'] / max(1, totals['reference_characters'])
    with (OUT / f'{split}_predictions.jsonl').open('w', encoding='utf-8') as stream:
        for item in predictions: stream.write(json.dumps(item, ensure_ascii=False) + '\n')
    return result

RUN_EXPLORATORY_TEST = False  # This split was consulted during development; it is not blind.
report = {'model_source': MODEL_ID, 'model_revision': MODEL_REVISION, 'seed': SEED, 'train_rows': len(order), 'mean_training_loss': sum(losses) / len(losses), 'manifest_sha256': MANIFEST_SHA256, 'validation': score('validation'), 'exploratory_test': score('test') if RUN_EXPLORATORY_TEST else None, 'source': 'ARTPARK-IISc/Vaani', 'license': 'CC-BY-4.0', 'reference_review_status': 'upstream provider labels; not native-speaker adjudicated', 'evaluation_note': 'Validation is a development measure against unreviewed provider references. Test is not run by default and has been consulted previously; neither supports a blind benchmark claim.'}
(OUT / 'run_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))
